Проблема - нельзя спарсить все вакансии, нельзя спарсить все компании

Идея -
1) Парсим по ключевому слову вакансии
2) Отсюда получаем какие-то точно айтишные вакансии
3) Дальше получаем уникальные айдишники кампаний
4) Дальше я забираю кампании и всю инфу по ним
5) Передаю айдишники, чтобы по ним забрать вакансии

In [6]:
import requests
import logging
import time
from datetime import datetime

In [7]:
def logging_get(url, headers, params={}):
    logging.basicConfig(level=logging.INFO, force=True, filename='api.log')

    logging.info(f"Запрос к {url}")
    logging.info(f"С параметрами {params}")
    response = requests.get(url, headers=headers, params=params)

    logging.info(f"Ответ: {response.status_code}")
    logging.info(f"Сейчас {datetime.now() }\n") # это по Гринвичу
    return response


In [8]:
secret_key = "v3.r.139742973.4b1b3e504a3e50866933ebf38a01b4040040b602.910ae5221ef50936ae42a82a671147a0991ef3e3"
headers = {"X-Api-App-Id": secret_key}


In [9]:
catalogue_code = 33

Запрос по апи к сущности каталога, чтобы достать для айтишной отрасли ее категории

Документация апи суперджоба https://api.superjob.ru/#catalogues:~:text=%7D%2C%20//...%0A%20%20%20%20%20%20%20%20%20%20%20%20%5D%0A%20%20%20%20%20%20%20%20%7D%2C%20//%20...%0A%5D-,%D0%9A%D0%B0%D1%82%D0%B5%D0%B3%D0%BE%D1%80%D0%B8%D0%B8%20%D0%BF%D0%BE%20%D0%BE%D1%82%D1%80%D0%B0%D1%81%D0%BB%D0%B8,-Resource%20information

In [10]:
url = f"https://api.superjob.ru/2.0/catalogues/parent/{catalogue_code}/"
params = {"count": 100}

response = logging_get(url, headers=headers, params=params)

response

<Response [200]>

In [11]:
data = response.json()
for item in data:
    print(item["key"], item["title_rus"])

651 AI
603 CRM-системы
627 Data Science
628 DevOps
629 SRE
36 Web-верстка
37 Администрирование баз данных
38 Аналитика
503 Внедрение и сопровождение ПО
40 Игровое ПО / Геймдев
41 Инжиниринг
42 Интернет, создание и поддержка сайтов
546 Информационная безопасность
620 Киберспорт
43 Компьютерная анимация и мультимедиа
44 Контент
604 Мобильная разработка
650 Нейросети / Искусственный интеллект
45 Оптимизация, SEO
46 Передача данных и доступ в интернет
47 Разработка и сопровождение банковского ПО
48 Разработка, программирование
49 Сетевые технологии
50 Системная интеграция
51 Системное администрирование
52 Системы автоматизированного проектирования (САПР)
53 Системы управления предприятием (ERP)
54 Сотовые, беспроводные технологии
55 Телекоммуникации и связь
56 Тестирование, QA
614 Техническая документация
57 Техническая поддержка
613 Управление продуктом
605 Управление проектами
630 Управление разработкой
59 Юзабилити
60 Другое
61 Начало карьеры, мало опыта


Получили ключевые слова, запишем их в наш словарик

In [12]:
#словарик
keywords = [
    'Python',
    'Java',
    'JavaScript',
    'Frontend',
    'Backend',
    'Fullstack',
    'DevOps',
    'Data Scientist',
    'Аналитик данных',
    'ML Engineer',
    'QA',
    'Тестировщик',
    'Product Manager',
    'Project Manager',
    'UI/UX',
    'Дизайнер',
    'Системный администратор',
    'Разработчик',
    'Программист',
    'Android',
    'iOS',
    'React',
    'Vue',
    'Angular',
    'Node.js',
    'PHP',
    'C++',
    'C#',
    '.NET',
    'Go',
    'Golang',
    'Kotlin',
    'Swift',
    'Data Engineer',
    'Business Analyst',
    'Scrum Master',
    'AI',
    'CRM-системы',
    'Data Science',
    'SRE',
    'Web-верстка',
    'Администрирование баз данных',
    'Аналитика',
    'Внедрение и сопровождение ПО',
    'Игровое ПО / Геймдев',
    'Инжиниринг',
    'Интернет, создание и поддержка сайтов',
    'Информационная безопасность',
    'Киберспорт',
    'Компьютерная анимация и мультимедиа',
    'Контент',
    'Мобильная разработка',
    'Нейросети / Искусственный интеллект',
    'Оптимизация, SEO',
    'Передача данных и доступ в интернет',
    'Разработка и сопровождение банковского ПО',
    'Разработка, программирование',
    'Сетевые технологии',
    'Системная интеграция',
    'Системное администрирование',
    'Системы автоматизированного проектирования (САПР)',
    'Системы управления предприятием (ERP)',
    'Сотовые, беспроводные технологии',
    'Телекоммуникации и связь',
    'Тестирование',
    'Техническая документация',
    'Техническая поддержка',
    'Управление продуктом',
    'Управление проектами',
    'Управление разработкой',
    'Юзабилити',
    'Начало карьеры'
]

In [13]:
def get_vacancies_by_keyword(keyword):

    url = 'https://api.superjob.ru/2.0/vacancies/'
    all_vacancies = []

    for page in range(5):
        params = {
            'keyword': keyword,
            'count': 100,
            'page': page,
            'c': 1,
            'catalogues': 33,
            'date_published_from': int(datetime(2024, 1, 1).timestamp())
        }

        try:
            response = logging_get(url, headers, params)
            data = response.json()
            objects = data.get('objects', [])

            if not objects:
                break

            all_vacancies.extend(objects)
            time.sleep(0.7)
        except:
            break

    return all_vacancies


all_vacancies_keywords = []

for keyword in keywords:
    vacancies = get_vacancies_by_keyword(keyword)
    if vacancies:
        all_vacancies_keywords.extend(vacancies)
        print(f"Найдено что-то по : {keyword}")
    else:
        print(f"Ничего не найдено по: {keyword}")

Найдено что-то по : Python
Найдено что-то по : Java
Найдено что-то по : JavaScript
Найдено что-то по : Frontend
Найдено что-то по : Backend
Найдено что-то по : Fullstack
Найдено что-то по : DevOps
Ничего не найдено по: Data Scientist
Найдено что-то по : Аналитик данных
Ничего не найдено по: ML Engineer
Найдено что-то по : QA
Найдено что-то по : Тестировщик
Найдено что-то по : Product Manager
Найдено что-то по : Project Manager
Ничего не найдено по: UI/UX
Найдено что-то по : Дизайнер
Найдено что-то по : Системный администратор
Найдено что-то по : Разработчик
Найдено что-то по : Программист
Найдено что-то по : Android
Найдено что-то по : iOS
Найдено что-то по : React
Найдено что-то по : Vue
Найдено что-то по : Angular
Найдено что-то по : Node.js
Найдено что-то по : PHP
Найдено что-то по : C++
Найдено что-то по : C#
Ничего не найдено по: .NET
Найдено что-то по : Go
Найдено что-то по : Golang
Найдено что-то по : Kotlin
Ничего не найдено по: Swift
Найдено что-то по : Data Engineer
Найдено ч

In [21]:
#достать айдишники
import pandas as pd
client_data = []

for vacancy in all_vacancies_keywords:
    client_id = vacancy.get('id_client')
    firm_name = vacancy.get('firm_name')

    if client_id:
        client_data.append({
            'id_client': client_id,
            'firm_name': firm_name
        })

df_clients_all = pd.DataFrame(client_data)

df_clients_all.drop_duplicates().to_csv('id_client_names_final.csv', index=False)
df_clients_all['id_client'].drop_duplicates().to_csv('id_client_final.csv', index=False)



In [22]:
df = pd.read_csv('id_client_names_final.csv')
df

,id_client,firm_name
0,4887815,Социальный фонд России
1,4942874,Точка банк
2,4944790,Лаборатория Касперского
3,4905942,OZON: Старт карьеры
4,4907133,МТС: СТАРТ КАРЬЕРЫ
...,...,...
180,4918378,"ФГБНУ ""РНЦХ ИМ. АКАД. Б. В. ПЕТРОВСКОГО"""
181,190718,Росгосстрах
182,4832769,ПК-Сервис
183,3797376,"ООО ""ВЕРТИКАЛЬ"""


In [23]:
#данные о компаниях

unique_company_ids = df['id_client'].dropna().unique()
unique_company_ids = [int(i) for i in unique_company_ids if i]
unique_company_ids

[4887815,
 4942874,
 4944790,
 4905942,
 4907133,
 11714,
 4932954,
 4944244,
 4944476,
 874575,
 4957734,
 14449,
 191494,
 4959545,
 4959764,
 2963005,
 4961511,
 4962080,
 3046437,
 4243646,
 3845930,
 4070787,
 237125,
 4959197,
 4905642,
 4904563,
 3595643,
 4346970,
 4859700,
 267938,
 4959724,
 4961508,
 4952362,
 89813,
 4922230,
 235578,
 4876809,
 173974,
 291920,
 13775,
 154674,
 4894541,
 4959207,
 3758350,
 860291,
 272717,
 3597464,
 108428,
 2914805,
 4953872,
 2464328,
 4564603,
 3551066,
 35359,
 2722945,
 4412294,
 2962986,
 3599075,
 1975482,
 71817,
 51445,
 2631007,
 3001379,
 4596048,
 314091,
 4891649,
 542741,
 1977505,
 15563,
 4763170,
 2756555,
 3959568,
 4273213,
 678972,
 2416642,
 4451620,
 335586,
 4918750,
 4140409,
 4853605,
 298376,
 4572294,
 3974341,
 4745650,
 2727860,
 4353589,
 2230116,
 196244,
 528217,
 4029196,
 2081428,
 514997,
 2063869,
 4954737,
 4398800,
 100593,
 4942854,
 4957726,
 4960349,
 3117918,
 4230892,
 4882733,
 4957980,
 35381

In [24]:
def get_company_info(company_id):
    url = f'https://api.superjob.ru/2.0/clients/{company_id}/'

    try:
        response = logging_get(url, headers)
        time.sleep(0.7)
        return response.json()

    except:
        return None

companies_data = []

for company_id in unique_company_ids:

    company_info = get_company_info(company_id)

    if company_info is not None:

        industries = company_info.get('industry', [])
        ids = [ind.get('id') for ind in industries]
        titles = [ind.get('title') for ind in industries]

        companies_data.append({
            'company_id': company_info.get('id'),
            'company_title': company_info.get('title'),
            'company_link': company_info.get('link'),
            'description': company_info.get('description'),
            'vacancy_count': company_info.get('vacancy_count'),
            'staff_count': company_info.get('staff_count'),
            'industry_ids': ', '.join(map(str, ids)),
            'industry_titles': ', '.join(titles),
            'client_logo': company_info.get('client_logo'),
            'is_blocked': company_info.get('is_blocked', False)
        })


In [25]:
# выгрузить данные о компаниях

df_companies = pd.DataFrame(companies_data)

df_companies.to_csv('chosen_companies_info.csv', index=False)


In [26]:
df_companies

,company_id,company_title,company_link,description,vacancy_count,staff_count,industry_ids,industry_titles,client_logo,is_blocked
0,4887815,Социальный фонд России,https://www.superjob.ru/clients/socialnyj-fond...,Фонд пенсионного и социального страхования Рос...,62,более 5000,,,https://public.superjob.ru/images/clients_logo...,False
1,4942874,Точка банк,https://www.superjob.ru/clients/tochka-bank-49...,Точка – полностью онлайновый банковский сервис...,5,менее 50,,,None,False
2,4944790,Лаборатория Касперского,https://www.superjob.ru/clients/laboratoriya-k...,«Лаборатория Касперского» – международная комп...,0,менее 50,,,None,False
3,4905942,OZON: Старт карьеры,https://www.superjob.ru/clients/ozon-start-kar...,Ozon — ведущая площадка электронной коммерции ...,10,более 5000,,,None,False
4,4907133,МТС: Старт карьеры,https://www.superjob.ru/clients/mts-start-kare...,Стажировки в МТС.\nПогрузись в цифровую экосис...,4,менее 50,,,None,False
...,...,...,...,...,...,...,...,...,...,...
161,4958617,Lasertech,https://www.superjob.ru/clients/lasertech-4958...,Lasertech — один из ведущих игроков на рынке п...,1,менее 50,,,None,False
162,4918378,"ФГБНУ ""РНЦХ ИМ. АКАД. Б. В. ПЕТРОВСКОГО""",https://www.superjob.ru/clients/fgbnu-rnch-im-...,Государственный научный центр Российской Федер...,5,1000 — 5000,,,https://public.superjob.ru/images/clients_logo...,False
163,190718,"ПАО СК ""Росгосстрах""",https://www.superjob.ru/clients/pao-sk-rosgoss...,"РОСГОССТРАХ — старейшая страховая компания, ос...",17,более 5000,,,https://public.superjob.ru/images/clients_logo...,False
164,4832769,ПК-Сервис,https://www.superjob.ru/clients/pk-servis-4832...,"Обслуживание компьютерной техники, заправка ка...",1,менее 50,,,None,False
